In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np
from sklearn.decomposition import PCA
import scipy.signal as signal
from IPython.display import Audio
from librosa.feature import melspectrogram
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
from sklearn.preprocessing import LabelEncoder
import os
import librosa
from dataset_bird import BirdDataset
from torch.utils.data import DataLoader, Dataset
import utils
import torch.nn as nn
import torch
# data augumentations
from utils import DataPipelines, CFG, data_transforms
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from train import train_model, evaluate
from models import BirdClassifier

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Importing data
train_df = pd.read_csv(CFG.train_csv)
test_df = pd.read_csv(CFG.test_csv)

# Train-Validation Split
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['primary_label'])

# Create Datasets
train_dataset = BirdDataset(train_df, augmentations=data_transforms())
val_dataset = BirdDataset(val_df, augmentations=data_transforms(mode='val'))

# Create DataLoaders 
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)



In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
model = BirdClassifier(num_classes=264).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
# learning rate scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [ ]:
train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=20, step_size=5, gamma=0.1)   